In [1]:
import numpy as np
from netCDF4 import Dataset
import matplotlib.pyplot as plt
#import cartopy.crs as crs
from cartopy.io.shapereader import Reader
from cartopy.feature import ShapelyFeature
from cartopy.feature import NaturalEarthFeature
import cartopy.crs as ccrs
from datetime import datetime, timedelta
from metpy.units import units
from metpy.calc import (dewpoint_from_specific_humidity, relative_humidity_from_specific_humidity, wind_speed, temperature_from_potential_temperature,
                    specific_humidity_from_mixing_ratio)
import pyart
import geopy.distance as distance
import haversine
from wrf import getvar
from metpy.interpolate import log_interpolate_1d


## You are using the Python ARM Radar Toolkit (Py-ART), an open source
## library for working with weather radar data. Py-ART is partly
## supported by the U.S. Department of Energy as part of the Atmospheric
## Radiation Measurement (ARM) Climate Research Facility, an Office of
## Science user facility.
##
## If you use this software to prepare a publication, please cite:
##
##     JJ Helmus and SM Collis, JORS 2016, doi: 10.5334/jors.119



In [2]:
#ncfile1 = Dataset('/glade/work/mawilson/DART_mpd/observations/utilities/threed_sphere/obs_epoch_100m.nc')
ncfile1 = Dataset('/glade/work/mawilson/DART_mpd/observations/utilities/threed_sphere/obs_epoch_100mIOP4.nc')
#ncfile1 = Dataset('/glade/work/mawilson/DART_mpd/observations/utilities/threed_sphere/obs_epoch_100mnew.nc')

location = ncfile1.variables['location'][:]
qc = ncfile1.variables['qc'][:]
obstype = ncfile1.variables['obs_type'][:]
obstypemd = ncfile1.variables['ObsTypesMetaData'][:]
obs_val = ncfile1.variables['observations'][:]
which_vert = ncfile1.variables['which_vert'][:]
times = ncfile1.variables['time'][:]
print(obstype)
qc_new = []
for i in range(len(qc)):
    qc_d = qc[i][0]
    qc_new.append(qc_d)
qc_new = np.asarray(qc_new)

[142 105 106 ...  26  27  28]


In [3]:
print(times)

[153965.99444444 153965.99444444 153965.99444444 ... 153966.41662037
 153966.41662037 153966.41662037]


In [6]:
otype=19

loc_T2 = location[obstype==otype, :]
qc_T2 = qc_new[obstype==otype]
obs_T2 = obs_val[obstype==otype, :]
lons_T2 = loc_T2[:,0]
lats_T2 = loc_T2[:,1]
elev_T2 = loc_T2[:,2]
time_T2 = times[obstype==otype]

print(len(obs_T2))

0


In [4]:
otype_s = []
obs_s = []
lon_s = []
lat_s = []
elev_s = []
error_s = []
time_s = []

minute_range = np.arange(180,605,5)
#minute_range = np.arange(595,605,5)

for mins in minute_range:
    #dt_start = datetime(2022,9,15,0,0)
    dt_start = datetime(2022,7,19,0,0)
    #dt_start = datetime(2021,6,4,0,0)
    
    dt = dt_start + timedelta(minutes=int(mins))
    print(dt)
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100IOP6/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    #wrfout = Dataset('/glade/derecho/scratch/mawilson/20210604_newcase/nature_100IOP4/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    #wrfout = Dataset('/glade/campaign/ral/aap/mawilson/nature_runs/JUNE/final_nature/wrfout_d02_2021-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    wrfout = Dataset('/glade/campaign/ral/aap/mawilson/nature_runs/IOP4/final_nature/wrfout_d02_2022-'+str(dt.isoformat()[5:7])+'-'+str(dt.isoformat()[8:10])+'_'+str(dt.isoformat()[11:13])+':'+str(dt.isoformat()[14:16])+':00')
    
    lon = wrfout.variables['XLONG']
    lat = wrfout.variables['XLAT']
    U10 = wrfout.variables['U10']
    V10 = wrfout.variables['U10']
    T2 = np.asarray(wrfout.variables['T2'])*units('K')
    T2F = T2 .to('degF')
    Q2 = np.asarray(wrfout.variables['Q2'])
    P2 = np.asarray(wrfout.variables['PSFC'][:]/100) * units('hPa')
    Td2 = dewpoint_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg'))
    RH2 = relative_humidity_from_specific_humidity(P2[0,:,:], T2[0,:,:], Q2[0,:,:]*units('kg/kg'))
    SPD10 = wind_speed(np.asarray(U10)*units('m/s'), np.asarray(V10)*units('m/s'))
    cloud=wrfout.variables['QCLOUD']
    T_z = np.asarray(getvar(wrfout, "tk"))
    p_z = np.asarray(getvar(wrfout, "pres"))
    q_zi = np.asarray(wrfout.variables['QVAPOR'][:])
    q_z = specific_humidity_from_mixing_ratio(q_zi)
    u_z, v_z = getvar(wrfout, 'uvmet')
    u_z = np.asarray(u_z)
    v_z = np.asarray(v_z)
    
    
    #otype = 107
    #otype = 105
    #otype = 106
    #otype = 108
    #otype = 142
    otype_list = [18,16,17]
    for otype in otype_list:
        loc_T2 = location[obstype==otype, :]
        qc_T2 = qc_new[obstype==otype]
        obs_T2 = obs_val[obstype==otype, :]
        lons_T2 = loc_T2[:,0]
        lats_T2 = loc_T2[:,1]
        elev_T2 = loc_T2[:,2]
        time_T2 = times[obstype==otype]
        lons_T2[lons_T2 > 180] = lons_T2[lons_T2 > 180] - 360
        
        #Convert WRF file time into same units as the obs_seq time
        dt_tot = (dt - datetime(1601,1,1)).total_seconds() / 86400
        time_diff = np.abs(dt_tot - time_T2)
        
        #Get obs within +- 2.5 minutes of each WRF file
        time_T3 = time_T2[time_diff<(150/86400)]
        loc_T3 = loc_T2[time_diff<(150/86400), :]
        qc_T3 = qc_T2[time_diff<(150/86400)]
        lons_T3 = lons_T2[time_diff<(150/86400)]
        lats_T3 = lats_T2[time_diff<(150/86400)]
        elev_T3 = elev_T2[time_diff<(150/86400)]
        obs_T3 = obs_T2[time_diff<(150/86400), :]
        
        if len(obs_T3)==0:
            print('NO OBS IN WINDOW')
        for k in range(len(lons_T3)):
            latp=lats_T3[k]
            lonp=lons_T3[k]
            #Get location for each ob in model land
            lon1d = np.ndarray.flatten(lon[0,:,:])
            lat1d = np.ndarray.flatten(lat[0,:,:])
            station = []
            points = []
            for i in range(len(lon1d)):
                points.append((lat1d[i],lon1d[i]))
                station.append((latp,lonp))
            dist = haversine.haversine_vector(station,points)
            dist2=dist.reshape(lon.shape[1],lon.shape[2])
            print(lon[0,:,:][np.where(dist2==np.min(dist2))])
            print(lat[0,:,:][np.where(dist2==np.min(dist2))])
            print(np.where(dist2==np.min(dist2)))
            st_xind = np.where(dist2==np.min(dist2))[0][0]
            st_yind = np.where(dist2==np.min(dist2))[1][0]
            print(elev_T3[k], 'elev')
            
            if otype == 18:
                p_point = p_z[:,st_xind,st_yind]
                t_point = T_z[:,st_xind,st_yind]
                T2_a = log_interpolate_1d(elev_T3[k], p_point, t_point)
                
            elif otype == 16:
                p_point = p_z[:,st_xind,st_yind]
                t_point = u_z[:,st_xind,st_yind]
                T2_a = log_interpolate_1d(elev_T3[k], p_point, t_point)
                
            elif otype == 17:
                p_point = p_z[:,st_xind,st_yind]
                t_point = v_z[:,st_xind,st_yind]
                T2_a = log_interpolate_1d(elev_T3[k], p_point, t_point)

            if np.isnan(T2_a):
                #If the observation is a nan or outside of the the interpolation bounds, skip it
                print('skipping nan ob')
                continue
            
            #If you want to change the error assumption, just change the scale in this line
            error = np.random.normal(loc=0.0, scale=np.sqrt(obs_T3[k,1]))
            #6/17/2024 MW adding code to limit added error to 1.5 times the error assumtion in DART
            if np.abs(error/4) > (np.sqrt(obs_T3[k,1])*1.0):
                error = (error / np.abs(error)) * (np.sqrt(obs_T3[k,1])*1.0)
            T2_b = T2_a + (error/4)
            print(T2_a, (error/4))
        
            #Append obs to arrays for writing to obs_seq file later
            otype_s.append(otype)
            obs_s.append(T2_b)
            lon_s.append(lonp)
            lat_s.append(latp)
            elev_s.append(elev_T3[k])
            error_s.append(obs_T3[k,1]/2)
            time_s.append(time_T3[k])

2022-07-19 03:00:00
[-85.42338]
[38.677452]
(array([6]), array([5]))
67029.27795001284 elev
[278.67329012] -0.19030665799230304
[-84.68475]
[39.025322]
(array([355]), array([580]))
46601.84544854423 elev
[263.06080294] -0.2894584530280554
[-84.616394]
[39.049988]
(array([380]), array([633]))
44816.75602739428 elev
[261.25609284] -0.21615319675280495
[-85.32746]
[38.731567]
(array([60]), array([80]))
68996.10886366865 elev
[280.22115341] -0.2885836497402469
[-85.37224]
[38.701515]
(array([30]), array([45]))
67685.55442952001 elev
[279.24828184] -0.46656982897297333
[-84.73265]
[38.98455]
(array([314]), array([543]))
48450.69707215526 elev
[264.71184096] 0.19041838160655727
[-84.77663]
[38.939743]
(array([269]), array([509]))
50474.61527938546 elev
[266.26026273] -0.18852838861658916
[-85.25064]
[38.787582]
(array([116]), array([140]))
71969.96138309117 elev
[282.22602576] -0.10072225265538351
[-85.28778]
[38.75958]
(array([88]), array([111]))
70237.76089775887 elev
[280.97522853] -0.116

/glade/derecho/scratch/mawilson/tmp/ipykernel_2971/1275381481.py:99: UserWarning: Interpolation point out of data bounds encountered
  T2_a = log_interpolate_1d(elev_T3[k], p_point, t_point)


[-84.98158]
[38.72742]
(array([56]), array([350]))
60705.66771040644 elev
[274.78155874] -0.0001224319266652812
[-85.10177]
[38.893497]
(array([222]), array([256]))
81278.5868876063 elev
[286.87024263] -0.08454016416623938
[-85.14032]
[38.86653]
(array([195]), array([226]))
78953.29677416314 elev
[285.62064563] 0.24647116810184613
[-85.42338]
[38.677452]
(array([6]), array([5]))
67029.27795001284 elev
[0.71052655] 1.5848559719198372
[-84.68475]
[39.025322]
(array([355]), array([580]))
46601.84544854423 elev
[-3.27621807] -0.025321033302285592
[-84.616394]
[39.049988]
(array([380]), array([633]))
44816.75602739428 elev
[-2.55367992] -0.35666215008901847
[-85.32746]
[38.731567]
(array([60]), array([80]))
68996.10886366865 elev
[3.05271476] -0.4667570265695551
[-85.37224]
[38.701515]
(array([30]), array([45]))
67685.55442952001 elev
[1.22687251] 0.291052356996734
[-84.73265]
[38.98455]
(array([314]), array([543]))
48450.69707215526 elev
[-3.33231868] -0.11207869665692052
[-84.77663]
[38.9

/glade/derecho/scratch/mawilson/tmp/ipykernel_2971/1275381481.py:104: UserWarning: Interpolation point out of data bounds encountered
  T2_a = log_interpolate_1d(elev_T3[k], p_point, t_point)


[-84.98158]
[38.72742]
(array([56]), array([350]))
60705.66771040644 elev
[-0.39185862] 1.8312283323771417
[-85.10177]
[38.893497]
(array([222]), array([256]))
81278.5868876063 elev
[3.85197394] -0.8500788642323605
[-85.14032]
[38.86653]
(array([195]), array([226]))
78953.29677416314 elev
[2.72007549] 0.09615337110450298
[-85.42338]
[38.677452]
(array([6]), array([5]))
67029.27795001284 elev
[0.29254789] 0.03908381887633526
[-84.68475]
[39.025322]
(array([355]), array([580]))
46601.84544854423 elev
[-7.23022527] 0.34712851908129116
[-84.616394]
[39.049988]
(array([380]), array([633]))
44816.75602739428 elev
[-9.11741324] 0.28561166397721555
[-85.32746]
[38.731567]
(array([60]), array([80]))
68996.10886366865 elev
[-0.17144841] -0.3016272733849022
[-85.37224]
[38.701515]
(array([30]), array([45]))
67685.55442952001 elev
[-2.61345526] -0.10337687114076594
[-84.73265]
[38.98455]
(array([314]), array([543]))
48450.69707215526 elev
[-4.78459792] -1.2818887769569196
[-84.77663]
[38.939743]
(

...

[-84.681946]
[39.058296]
(array([388]), array([582]))
98426.4173831982 elev
skipping nan ob
[-84.717995]
[39.05845]
(array([388]), array([554]))
96863.3728363273 elev
[297.55886113] 1.040063547969314
[-85.00521]
[38.98831]
(array([317]), array([331]))
85990.3295874229 elev
[-0.38322942] 0.48783622310934727
[-84.966545]
[39.01323]
(array([342]), array([361]))
87794.50326500641 elev
[-0.24060271] -0.08464491614493835
[-84.930435]
[39.03714]
(array([366]), array([389]))
89238.11645796885 elev
[0.34357425] 0.7638137568542462
[-84.893036]
[39.058033]
(array([387]), array([418]))
89173.06867501476 elev
[0.45851489] -0.6661212849329088
[-84.85314]
[39.056923]
(array([386]), array([449]))
90273.17127208646 elev
[0.96376367] -0.480249977039603
[-84.81324]
[39.0568]
(array([386]), array([480]))
92160.76318651349 elev
[2.09341509] -0.034543838363207464
[-84.74761]
[39.05656]
(array([386]), array([531]))
95389.14648383862 elev
[4.06813279] 0.6412948114718195
[-84.77849]
[39.05668]
(array([386]), a

/glade/derecho/scratch/mawilson/tmp/ipykernel_2971/1275381481.py:104: UserWarning: Interpolation point out of data bounds encountered
  T2_a = log_interpolate_1d(elev_T3[k], p_point, t_point)


[-84.681946]
[39.058296]
(array([388]), array([582]))
98426.4173831982 elev
skipping nan ob
[-84.717995]
[39.05845]
(array([388]), array([554]))
96863.3728363273 elev
[5.05838095] 0.06450825814338954
[-85.00521]
[38.98831]
(array([317]), array([331]))
85990.3295874229 elev
[-3.37712684] 0.55433037399735
[-84.966545]
[39.01323]
(array([342]), array([361]))
87794.50326500641 elev
[-2.08541562] 0.04260772796176162
[-84.930435]
[39.03714]
(array([366]), array([389]))
89238.11645796885 elev
[-0.75792699] -0.6327253028355196
[-84.893036]
[39.058033]
(array([387]), array([418]))
89173.06867501476 elev
[-1.10415811] 0.35791208946259173
[-84.85314]
[39.056923]
(array([386]), array([449]))
90273.17127208646 elev
[-1.11496086] 0.3026389235732415
[-84.81324]
[39.0568]
(array([386]), array([480]))
92160.76318651349 elev
[0.38845255] -0.09335553725785287
[-84.74761]
[39.05656]
(array([386]), array([531]))
95389.14648383862 elev
[0.93943365] 0.39146067031527343
[-84.77849]
[39.05668]
(array([386]), a

/glade/derecho/scratch/mawilson/tmp/ipykernel_2971/1275381481.py:109: UserWarning: Interpolation point out of data bounds encountered
  T2_a = log_interpolate_1d(elev_T3[k], p_point, t_point)


[-84.681946]
[39.058296]
(array([388]), array([582]))
98426.4173831982 elev
skipping nan ob
[-84.717995]
[39.05845]
(array([388]), array([554]))
96863.3728363273 elev
[0.75389191] 0.5680921608160436
2022-07-19 05:10:00
[-85.379745]
[39.14025]
(array([469]), array([40]))
23738.65935816362 elev
[229.81768571] -0.2352609761822539
[-85.379745]
[39.14025]
(array([469]), array([40]))
23738.65935816362 elev
[-6.68856762] 0.25536386334329275
[-85.379745]
[39.14025]
(array([469]), array([40]))
23738.65935816362 elev
[-13.06534232] -0.25738403395166815
2022-07-19 05:15:00
NO OBS IN WINDOW
NO OBS IN WINDOW
NO OBS IN WINDOW
2022-07-19 05:20:00
[-83.885445]
[39.462856]
(array([799]), array([1195]))
30043.40118202101 elev
[240.94260881] -0.1712663159463226
[-83.885445]
[39.462856]
(array([799]), array([1195]))
30043.40118202101 elev
[-8.60653033] 0.687922237790719
[-83.885445]
[39.462856]
(array([799]), array([1195]))
30043.40118202101 elev
[-13.31588526] -0.4688722709762469
2022-07-19 05:25:00
[-84

In [5]:
#Convert lats and lons to radians for DART, because why not
lon_DART = np.radians(np.asarray(lon_s))
lat_DART = np.radians(np.asarray(lat_s))

lon_DART = np.where(lon_DART > 0.0, lon_DART, lon_DART+(2.0*np.pi))

#Convert time into DART format. This is hacky now, improve later
#day_DART = 154024
day_DART = 153966
#day_DART = 153556

seconds_DART = (np.asarray(time_s) - day_DART) * 86400

In [6]:
#Sort everything in time order
inds_time = seconds_DART.argsort()
print(seconds_DART)
print(seconds_DART[inds_time])
seconds_DART1 = seconds_DART[inds_time]
seconds_DART1[seconds_DART1 < 0] = 0
obs_s1 = np.asarray(obs_s)[inds_time]
lon_DART1 = lon_DART[inds_time]
lat_DART1 = lat_DART[inds_time]
elev_s1 = np.asarray(elev_s)[inds_time]
otype_s1 = np.asarray(otype_s)[inds_time]
error_s1 = np.asarray(error_s)[inds_time]

[10679.99999961 10679.99999961 10679.99999961 10740.00000106
 10740.00000106 10740.00000106 10740.00000106 10800.
 10800.         10800.         10800.         10859.99999894
 10859.99999894 10859.99999894 10859.99999894 10920.00000039
 10920.00000039 10920.00000039 10679.99999961 10679.99999961
 10679.99999961 10740.00000106 10740.00000106 10740.00000106
 10740.00000106 10800.         10800.         10800.
 10800.         10859.99999894 10859.99999894 10859.99999894
 10859.99999894 10920.00000039 10920.00000039 10920.00000039
 10679.99999961 10679.99999961 10679.99999961 10740.00000106
 10740.00000106 10740.00000106 10740.00000106 10800.
 10800.         10800.         10800.         10859.99999894
 10859.99999894 10859.99999894 10859.99999894 10920.00000039
 10920.00000039 10920.00000039 10979.99999933 10979.99999933
 10979.99999933 11040.00000078 11040.00000078 11099.99999972
 11099.99999972 11099.99999972 11099.99999972 11160.00000117
 11220.00000011 11220.00000011 11220.00000011 10

In [7]:
for bigfoot in [1,2]:
    print(bigfoot)

    #Write the simulated obs out to an obs_seq file
    #filename = 'SIM_ACARS_IOP6_finalhalferr'
    #filename = 'SIM_ACARS_IOP4_finalhalferr'
    #filename = 'SIM_ACARS_JUNE_finalhalferr'
    filename = 'SIM_ACARS_IOP4_finalhalferr'
    
    fi = open(filename, "w")
    fi.write(" obs_sequence\n")
    fi.write("obs_kind_definitions\n")
    fi.write("    %d \n" %(3))
    fi.write("    %d          %s   \n" %(18, 'ACARS_TEMPERATURE'))
    fi.write("    %d          %s   \n" %(16, 'ACARS_U_WIND_COMPONENT'))
    fi.write("    %d          %s   \n" %(17, 'ACARS_V_WIND_COMPONENT'))
    
    fi.write("  num_copies:            %d  num_qc:            %d\n" % (1,1))
    fi.write(" num_obs:       %d  max_num_obs:       %d\n" % (len(obs_s1), len(obs_s1)))
    fi.write("MADIS observation\n")
    fi.write("Data QC\n")
    
    fi.write("  first:            %d  last:       %d\n" % (1, len(obs_s1)))
    
    for q in range(len(obs_s1)):
    
        fi.write(" OBS            %d\n" % (q+1) )
        fi.write("   %20.14f\n" % obs_s1[q] )
        fi.write("   %20.14f\n" % 1.0 )
    
        if q+1 == 1:
            fi.write(" %d %d %d\n" % (-1, q+2, -1) ) #First obs
        elif q+1 == len(obs_s1):
            fi.write(" %d %d %d\n" % (q, -1, -1) ) #Last obs
        else:
            fi.write(" %d %d %d\n" % (q, q+2, -1) )
    
        fi.write("obdef\n")
        fi.write("loc3d\n")
        fi.write("    %20.14f          %20.14f          %20.14f     %d\n" %
                           (lon_DART1[q], lat_DART1[q], elev_s1[q], 2))
        fi.write("kind\n")
    
        fi.write("     %d     \n" % otype_s1[q])
    
        fi.write("    %d          %d     \n" % (seconds_DART1[q], day_DART))
    
        fi.write("    %20.14f  \n" % error_s1[q])

1
2


/glade/derecho/scratch/mawilson/tmp/ipykernel_2971/2621984341.py:28: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  fi.write("   %20.14f\n" % obs_s1[q] )


In [8]:
print(obs_s)

[array([278.48298347]), array([262.77134449]), array([261.03993965]), array([279.93256976]), array([278.78171201]), array([264.90225935]), array([266.07173434]), array([282.12530351]), array([280.85922088]), array([268.01938593]), array([269.48139978]), array([271.8955564]), array([273.8073926]), array([284.66194137]), array([283.15480556]), array([274.78143631]), array([286.78570247]), array([285.8671168]), array([2.29538252]), array([-3.30153911]), array([-2.91034207]), array([2.58595774]), array([1.51792486]), array([-3.44439738]), array([-4.53200585]), array([1.51169035]), array([-0.27950999]), array([-3.47888396]), array([-4.40880441]), array([-2.29594101]), array([-1.25021336]), array([1.22932475]), array([-0.00640985]), array([1.43936971]), array([3.00189508]), array([2.81622886]), array([0.33163171]), array([-6.88309675]), array([-8.83180158]), array([-0.47307568]), array([-2.71683213]), array([-6.06648669]), array([-2.75680356]), array([-2.10055006]), array([-2.01773233]), arr